In [1]:
# -*- coding: utf-8 -*-
"""Meta_SPINN_Advection_with_Corrector.ipynb

Dynamic-basis meta-SPINN predictor + frozen-feature physics corrector for
1D periodic constant advection with Gaussian initial condition.

Problem
-------
    u_t + u_x = 0,   x in [0,1), t in [0,1]
with periodic BC and periodic Gaussian IC
    u(x,0) = exp( - d_per(x, x0)^2 / (2 nu^2) )

Predictor
---------
The predictor keeps the original dynamic periodic RBF representation.

Corrector
---------
For a test task p=[x0,nu], the corrector builds a frozen spacetime periodic
RBF dictionary from:
    (i) predictor-inherited dynamic centers,
    (ii) residual/gradient refinement centers,
    (iii) a modest periodic background grid.
It then solves a ridge-regularized linear collocation system for

    u_corr(x,t) = u0(x) + t * sum_j c_j phi_j(x,t)

which satisfies the initial condition exactly and tightens the PDE residual.
"""

import os
import csv
import math
import random
import argparse
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# ============================================================
# Reproducibility
# ============================================================
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# Utilities
# ============================================================
def get_device(use_gpu_flag: bool) -> torch.device:
    if use_gpu_flag and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def periodic_distance_torch(x: torch.Tensor, mu: torch.Tensor) -> torch.Tensor:
    return torch.remainder(x - mu + 0.5, 1.0) - 0.5


def periodic_distance_numpy(x: np.ndarray, mu: np.ndarray) -> np.ndarray:
    return np.remainder(x - mu + 0.5, 1.0) - 0.5


def periodic_gaussian_torch(x: torch.Tensor, x0: torch.Tensor, nu: torch.Tensor) -> torch.Tensor:
    d = periodic_distance_torch(x, x0)
    return torch.exp(-(d ** 2) / (2.0 * nu ** 2))


def periodic_gaussian_numpy(x: np.ndarray, x0: float, nu: float) -> np.ndarray:
    d = periodic_distance_numpy(x, x0)
    return np.exp(-(d ** 2) / (2.0 * nu ** 2))


def periodic_gaussian_dx_numpy(x: np.ndarray, x0: float, nu: float) -> np.ndarray:
    d = periodic_distance_numpy(x, x0)
    u0 = np.exp(-(d ** 2) / (2.0 * nu ** 2))
    return -(d / (nu ** 2)) * u0


def fourier_features(t: torch.Tensor, n_freq: int) -> torch.Tensor:
    feats = [t]
    for k in range(1, n_freq + 1):
        feats.append(torch.sin(2.0 * math.pi * k * t))
        feats.append(torch.cos(2.0 * math.pi * k * t))
    return torch.cat(feats, dim=-1)


def clip_numpy(x, lo, hi):
    return np.minimum(np.maximum(x, lo), hi)


# ============================================================
# Predictor model
# ============================================================
class DynamicMetaSPINN(nn.Module):
    def __init__(
        self,
        M: int = 32,
        hidden_task: int = 64,
        hidden_time: int = 96,
        n_fourier: int = 4,
        h_min: float = 0.015,
        h_max: float = 0.25,
        vel_scale: float = 0.25,
    ) -> None:
        super().__init__()
        self.M = M
        self.n_fourier = n_fourier
        self.h_min = h_min
        self.h_max = h_max
        self.vel_scale = vel_scale

        self.task_net = nn.Sequential(
            nn.Linear(2, hidden_task),
            nn.Tanh(),
            nn.Linear(hidden_task, hidden_task),
            nn.Tanh(),
        )

        self.init_head = nn.Linear(hidden_task, 4 * M)
        in_dyn = hidden_task + (1 + 2 * n_fourier)
        self.dyn_net = nn.Sequential(
            nn.Linear(in_dyn, hidden_time),
            nn.Tanh(),
            nn.Linear(hidden_time, hidden_time),
            nn.Tanh(),
            nn.Linear(hidden_time, 4 * M),
        )

    def normalize_p(self, p: torch.Tensor) -> torch.Tensor:
        mean = p.new_tensor([0.5, 0.075])
        std = p.new_tensor([0.3, 0.03])
        return (p - mean) / std

    def basis_params(self, p: torch.Tensor, t: torch.Tensor):
        B, N, _ = t.shape
        p_norm = self.normalize_p(p)
        emb = self.task_net(p_norm)

        init_out = self.init_head(emb).view(B, 4, self.M)
        gate_logits = init_out[:, 0, :]
        xi0_raw = init_out[:, 1, :]
        h0_raw = init_out[:, 2, :]
        a0_raw = init_out[:, 3, :]

        gate = torch.sigmoid(gate_logits)
        xi0 = torch.sigmoid(xi0_raw)
        h0 = self.h_min + (self.h_max - self.h_min) * torch.sigmoid(h0_raw)
        a0 = torch.tanh(a0_raw)

        tf = fourier_features(t.reshape(B * N, 1), self.n_fourier).reshape(B, N, -1)
        emb_rep = emb.unsqueeze(1).expand(-1, N, -1)
        dyn_in = torch.cat([emb_rep, tf], dim=-1)
        dyn_out = self.dyn_net(dyn_in).view(B, N, 4, self.M)

        dxi_raw = dyn_out[:, :, 0, :]
        dh_raw = dyn_out[:, :, 1, :]
        alpha_raw = dyn_out[:, :, 2, :]
        vel_raw = dyn_out[:, :, 3, :]

        xi = torch.remainder(
            xi0.unsqueeze(1) + 0.35 * torch.tanh(dxi_raw) + self.vel_scale * t * torch.tanh(vel_raw),
            1.0,
        )
        h = torch.clamp(
            h0.unsqueeze(1) * (1.0 + 0.35 * torch.tanh(dh_raw)),
            min=self.h_min,
            max=self.h_max,
        )
        alpha = a0.unsqueeze(1) + 1.0 * torch.tanh(alpha_raw)
        gate_full = gate.unsqueeze(1).expand(-1, N, -1)
        return alpha, xi, h, gate_full

    def forward(self, p: torch.Tensor, xt: torch.Tensor):
        x = xt[:, :, 0:1]
        t = xt[:, :, 1:2]
        alpha, xi, h, gate = self.basis_params(p, t)
        d = periodic_distance_torch(x, xi)
        phi = torch.exp(-(d ** 2) / (2.0 * h ** 2))
        u = torch.sum((gate * alpha) * phi, dim=-1)
        return u, (gate, alpha, xi, h)


# ============================================================
# Sampling
# ============================================================
def sample_p_batch(B: int, device: torch.device, x0_range=(0.2, 0.8), nu_range=(0.03, 0.12)) -> torch.Tensor:
    x0 = x0_range[0] + (x0_range[1] - x0_range[0]) * torch.rand(B, 1, device=device)
    u = torch.rand(B, 1, device=device)
    log_nu_min = math.log10(nu_range[0])
    log_nu_max = math.log10(nu_range[1])
    nu = 10.0 ** (log_nu_min + (log_nu_max - log_nu_min) * u)
    return torch.cat([x0, nu], dim=1)


def sample_interior_points_batch(B: int, N: int, device: torch.device):
    x = torch.rand(B, N, 1, device=device)
    t = torch.rand(B, N, 1, device=device)
    return torch.cat([x, t], dim=-1)


def sample_ic_points_batch(B: int, N: int, device: torch.device):
    x = torch.rand(B, N, 1, device=device)
    t = torch.zeros(B, N, 1, device=device)
    return torch.cat([x, t], dim=-1)


def sample_periodic_points_batch(B: int, N: int, device: torch.device):
    t = torch.rand(B, N, 1, device=device)
    xL = torch.zeros(B, N, 1, device=device)
    xR = torch.ones(B, N, 1, device=device)
    return torch.cat([xL, t], dim=-1), torch.cat([xR, t], dim=-1)


def sample_near_ic_points_batch(B: int, N: int, device: torch.device, t_max: float = 0.15):
    x = torch.rand(B, N, 1, device=device)
    t = t_max * torch.rand(B, N, 1, device=device)
    return torch.cat([x, t], dim=-1)


# ============================================================
# PDE residual for predictor
# ============================================================
def advection_residual_batch(model: nn.Module, p: torch.Tensor, xt_int: torch.Tensor) -> torch.Tensor:
    xt_int = xt_int.clone().detach().requires_grad_(True)
    u, _ = model(p, xt_int)
    grads = torch.autograd.grad(
        outputs=u,
        inputs=xt_int,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True,
    )[0]
    return grads[:, :, 1] + grads[:, :, 0]


# ============================================================
# Evaluation helpers
# ============================================================
def exact_solution(x, t, x0, nu):
    s = np.remainder(x - t, 1.0)
    return periodic_gaussian_numpy(s, x0, nu)


def predictor_field_numpy(model, device, x0_test, nu_test, X, T):
    xt_grid = np.stack([X.ravel(), T.ravel()], axis=1)
    xt_grid_t = torch.tensor(xt_grid, dtype=torch.float32, device=device).view(1, -1, 2)
    p_test = torch.tensor([[x0_test, nu_test]], dtype=torch.float32, device=device)
    model.eval()
    with torch.no_grad():
        u_pred_flat, _ = model(p_test, xt_grid_t)
    return u_pred_flat.cpu().numpy().reshape(X.shape)


def predictor_derivatives_numpy(model, device, x0_test, nu_test, X, T):
    pts = np.stack([X.ravel(), T.ravel()], axis=1)
    xt = torch.tensor(pts, dtype=torch.float32, device=device).view(1, -1, 2)
    xt.requires_grad_(True)
    p = torch.tensor([[x0_test, nu_test]], dtype=torch.float32, device=device)
    model.eval()
    u, _ = model(p, xt)
    grads = torch.autograd.grad(
        outputs=u,
        inputs=xt,
        grad_outputs=torch.ones_like(u),
        create_graph=False,
        retain_graph=False,
    )[0][0]
    u_np = u.detach().cpu().numpy().reshape(X.shape)
    ux_np = grads[:, 0].detach().cpu().numpy().reshape(X.shape)
    ut_np = grads[:, 1].detach().cpu().numpy().reshape(X.shape)
    return u_np, ux_np, ut_np


# ============================================================
# Corrector geometry and solve
# ============================================================


@dataclass
class CorrectorConfig:
    K_t: int = 12
    dyn_per_time: int = 4
    bg_x: int = 5
    bg_t: int = 4
    probe_x: int = 240
    peak_probe_x: int = 320
    local_peaks_per_time: int = 1
    residual_hotspots_per_time: int = 1
    fine_res_x: int = 720
    sigma_t_dyn_scale: float = 1.00
    sigma_t_local_scale: float = 0.85
    sigma_t_res_scale: float = 0.78
    sigma_t_bg_scale: float = 1.15
    dyn_width_inflate: float = 1.20
    bg_sigma_x_factor: float = 1.15
    local_width_factors_peak: tuple = (0.85, 1.10, 1.35)
    local_width_factors_shoulder: tuple = (0.95, 1.20)
    residual_width_factors: tuple = (0.90, 1.15)
    shoulder_shift_factor: float = 0.90
    ridge_list: tuple = (5e-4, 1e-3, 5e-3, 1e-2, 2e-2)
    alpha_list: tuple = (1.25, 1.0, 0.75, 0.5, 0.25, 0.0)
    residual_weight_floor: float = 0.10
    support_power: float = 1.10
    corr_shrink: float = 1.25
    coeff_shrink: float = 0.85
    colloc_x: int = 150
    colloc_t: int = 105
    probe_eval_x: int = 200
    probe_eval_t: int = 90
    fine_residual_boost: float = 1.0


def choose_corrector_config(nu_test: float) -> CorrectorConfig:
    if nu_test <= 0.03:
        return CorrectorConfig(
            K_t=18, dyn_per_time=5, bg_x=5, bg_t=4, probe_x=320, peak_probe_x=480,
            local_peaks_per_time=1, residual_hotspots_per_time=2, fine_res_x=1200,
            sigma_t_dyn_scale=0.95, sigma_t_local_scale=0.80, sigma_t_res_scale=0.72,
            dyn_width_inflate=1.14, colloc_x=175, colloc_t=125, probe_eval_x=240, probe_eval_t=110,
            residual_weight_floor=0.04, support_power=1.55, corr_shrink=0.75, coeff_shrink=0.55,
            ridge_list=(2e-4, 5e-4, 1e-3, 5e-3, 1e-2),
            alpha_list=(1.5, 1.25, 1.0, 0.75, 0.5, 0.25),
            fine_residual_boost=1.35
        )
    if nu_test <= 0.06:
        return CorrectorConfig(
            K_t=15, dyn_per_time=4, bg_x=5, bg_t=4, probe_x=280, peak_probe_x=420,
            local_peaks_per_time=1, residual_hotspots_per_time=2, fine_res_x=960,
            sigma_t_dyn_scale=0.98, sigma_t_local_scale=0.82, sigma_t_res_scale=0.75,
            dyn_width_inflate=1.17, colloc_x=162, colloc_t=115, probe_eval_x=218, probe_eval_t=96,
            residual_weight_floor=0.06, support_power=1.35, corr_shrink=0.95, coeff_shrink=0.70,
            ridge_list=(2e-4, 5e-4, 1e-3, 5e-3, 1e-2),
            alpha_list=(1.35, 1.1, 0.9, 0.7, 0.5, 0.25),
            fine_residual_boost=1.15
        )
    return CorrectorConfig(
        K_t=12, dyn_per_time=3, bg_x=5, bg_t=4, probe_x=240, peak_probe_x=340,
        local_peaks_per_time=1, residual_hotspots_per_time=1, fine_res_x=720,
        sigma_t_dyn_scale=1.00, sigma_t_local_scale=0.85, sigma_t_res_scale=0.78,
        dyn_width_inflate=1.20, colloc_x=145, colloc_t=100, probe_eval_x=190, probe_eval_t=84,
        residual_weight_floor=0.08, support_power=1.15, corr_shrink=1.10, coeff_shrink=0.80,
        ridge_list=(2e-4, 5e-4, 1e-3, 5e-3, 1e-2),
        alpha_list=(1.25, 1.0, 0.8, 0.6, 0.4, 0.2),
        fine_residual_boost=1.0
    )


def periodic_spacetime_basis_numpy(x, t, mx, mt, sx, st):
    dx = periodic_distance_numpy(x[:, None], mx[None, :])
    dt = t[:, None] - mt[None, :]
    phi = np.exp(-0.5 * (dx / sx[None, :]) ** 2 - 0.5 * (dt / st[None, :]) ** 2)
    phi_x = -(dx / (sx[None, :] ** 2)) * phi
    phi_t = -(dt / (st[None, :] ** 2)) * phi
    return phi, phi_x, phi_t


def unique_append(center_list, width_list, time_list, twidth_list, x_new, sx_new, t_new, st_new, min_sep):
    for x_old, t_old in zip(center_list, time_list):
        dx = abs((x_new - x_old + 0.5) % 1.0 - 0.5)
        if dx < min_sep and abs(t_new - t_old) < 0.6 * max(st_new, 1e-12):
            return
    center_list.append(float(x_new % 1.0))
    width_list.append(float(sx_new))
    time_list.append(float(t_new))
    twidth_list.append(float(st_new))


def extract_dynamic_centers_from_predictor(model, device, x0_test, nu_test, cfg: CorrectorConfig):
    t_vals = np.linspace(0.0, 1.0, cfg.K_t)
    dtg = t_vals[1] - t_vals[0] if cfg.K_t > 1 else 1.0
    p = torch.tensor([[x0_test, nu_test]], dtype=torch.float32, device=device)
    t = torch.tensor(t_vals, dtype=torch.float32, device=device).view(1, cfg.K_t, 1)
    with torch.no_grad():
        alpha, xi, h, gate = model.basis_params(p, t)
    score = np.abs((alpha[0] * gate[0]).cpu().numpy())
    xi = xi[0].cpu().numpy()
    h = h[0].cpu().numpy()

    x_centers, sx_list, t_centers, st_list = [], [], [], []
    x_probe_dx = 1.0 / max(cfg.probe_x, 2)
    sx_min = max(1.5 * x_probe_dx, 0.40 * nu_test)
    sx_max = min(0.24, 2.75 * nu_test)
    st = max(cfg.sigma_t_dyn_scale * dtg, 2e-3)

    for k, tk in enumerate(t_vals):
        idx = np.argsort(-score[k])[:cfg.dyn_per_time]
        for j in idx:
            sx = clip_numpy(cfg.dyn_width_inflate * h[k, j], sx_min, sx_max)
            unique_append(x_centers, sx_list, t_centers, st_list, xi[k, j], sx, tk, st, min_sep=0.55 * sx)

    return np.array(x_centers), np.array(t_centers), np.array(sx_list), np.array(st_list)




def local_periodic_spacing(points: np.ndarray, fallback: float) -> np.ndarray:
    points = np.asarray(points, dtype=float).reshape(-1)
    n = len(points)
    if n == 0:
        return np.array([], dtype=float)
    if n == 1:
        return np.array([float(fallback)], dtype=float)
    order = np.argsort(points)
    xs = points[order]
    right = np.roll(xs, -1) - xs
    right[-1] = (xs[0] + 1.0) - xs[-1]
    left = xs - np.roll(xs, 1)
    left[0] = xs[0] + 1.0 - xs[-1]
    spacing_sorted = np.minimum(left, right)
    spacing = np.empty_like(spacing_sorted)
    spacing[order] = spacing_sorted
    spacing = np.maximum(spacing, fallback)
    return spacing

def find_periodic_local_maxima(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values).reshape(-1)
    if values.size == 0:
        return np.array([], dtype=int)
    prev_v = np.roll(values, 1)
    next_v = np.roll(values, -1)
    idx = np.where((values >= prev_v) & (values >= next_v))[0]
    if idx.size == 0:
        idx = np.array([int(np.argmax(values))], dtype=int)
    return idx


def estimate_peak_sigma_from_profile(x_grid: np.ndarray, profile: np.ndarray, peak_idx: int, nu_ref: float) -> float:
    x_grid = np.asarray(x_grid).reshape(-1)
    profile = np.asarray(profile).reshape(-1)
    n = len(x_grid)
    dx = 1.0 / max(n, 1)
    if n == 0:
        return max(2.0 * dx, 0.5 * max(nu_ref, dx))

    shift = n // 2 - int(peak_idx)
    y = np.roll(profile, shift)
    c = n // 2
    peak_val = float(max(y[c], 1e-12))
    half = 0.5 * peak_val

    left = c
    while left > 0 and y[left] >= half:
        left -= 1
    right = c
    while right < n - 1 and y[right] >= half:
        right += 1

    fwhm = max((right - left) * dx, 2.0 * dx)
    sigma = fwhm / (2.0 * math.sqrt(2.0 * math.log(2.0)))
    sigma_min = max(1.25 * dx, 0.20 * max(nu_ref, dx))
    sigma_max = min(0.20, 2.25 * max(nu_ref, 2.0 * dx))
    return clip_numpy(sigma, sigma_min, sigma_max)


def extract_local_multiscale_centers_from_predictor(model, device, x0_test, nu_test, cfg: CorrectorConfig):
    t_vals = np.linspace(0.0, 1.0, cfg.K_t)
    dtg = t_vals[1] - t_vals[0] if cfg.K_t > 1 else 1.0
    x_probe = np.linspace(0.0, 1.0, cfg.peak_probe_x, endpoint=False)
    dxp = 1.0 / max(cfg.peak_probe_x, 2)

    x_centers, sx_list, t_centers, st_list = [], [], [], []
    st = max(cfg.sigma_t_local_scale * dtg, 2e-3)

    for tk in t_vals:
        Xk, Tk = np.meshgrid(x_probe, np.array([tk]), indexing="ij")
        Uk = predictor_field_numpy(model, device, x0_test, nu_test, Xk, Tk).reshape(-1)
        mag = np.abs(Uk)
        peak_candidates = find_periodic_local_maxima(mag)
        peak_candidates = sorted(peak_candidates.tolist(), key=lambda i: mag[i], reverse=True)
        if len(peak_candidates) == 0:
            peak_candidates = [int(np.argmax(mag))]

        selected = []
        amp_max = float(np.max(mag)) + 1e-12
        min_sep = max(4 * dxp, 0.35 * max(nu_test, dxp))
        for idx in peak_candidates:
            if mag[idx] < 0.20 * amp_max:
                continue
            xpk = float(x_probe[idx])
            if all(abs(periodic_distance_numpy(np.array([xpk]), x_probe[j])[0]) > min_sep for j in selected):
                selected.append(idx)
            if len(selected) >= cfg.local_peaks_per_time:
                break
        if len(selected) == 0:
            selected = [int(np.argmax(mag))]

        for idx in selected:
            xpk = float(x_probe[idx])
            sigma_hat = estimate_peak_sigma_from_profile(x_probe, mag, idx, nu_test)
            sigma_hat = clip_numpy(sigma_hat, max(1.1 * dxp, 0.22 * max(nu_test, dxp)), min(0.18, 2.0 * max(nu_test, 2 * dxp)))

            shift = cfg.shoulder_shift_factor * sigma_hat
            x_positions = [xpk] + [float((xpk + sign * shift) % 1.0) for sign in (-1.0, 1.0)]
            spacing = local_periodic_spacing(np.array(x_positions), fallback=max(2.0 * dxp, 0.22 * max(nu_test, dxp)))
            base_peak = clip_numpy(1.05 * spacing[0], max(1.4 * dxp, 0.18 * max(nu_test, dxp)), min(0.20, 0.95 * max(nu_test, 2 * dxp), 1.9 * sigma_hat))
            shoulder_base = clip_numpy(1.05 * spacing[1], max(1.4 * dxp, 0.20 * max(nu_test, dxp)), min(0.22, 1.15 * max(nu_test, 2 * dxp), 2.1 * sigma_hat))

            # Peak-centered multiscale atoms with spacing-based overlap control.
            for fac in cfg.local_width_factors_peak:
                sx = clip_numpy(fac * base_peak, max(1.4 * dxp, 0.18 * max(nu_test, dxp)), min(0.24, 1.35 * max(nu_test, 2 * dxp), 2.2 * base_peak))
                unique_append(x_centers, sx_list, t_centers, st_list, xpk, sx, tk, st, min_sep=0.35 * base_peak)

            # Shoulder atoms allow small relocalization and sharpening without characteristic priors.
            for sign, idx_sp in zip((-1.0, 1.0), (1, 2)):
                xsh = float((xpk + sign * shift) % 1.0)
                shoulder_base = clip_numpy(1.05 * spacing[idx_sp], max(1.4 * dxp, 0.20 * max(nu_test, dxp)), min(0.22, 1.10 * max(nu_test, 2 * dxp), 2.0 * sigma_hat))
                for fac in cfg.local_width_factors_shoulder:
                    sx = clip_numpy(fac * shoulder_base, max(1.4 * dxp, 0.20 * max(nu_test, dxp)), min(0.24, 1.30 * max(nu_test, 2 * dxp), 2.1 * shoulder_base))
                    unique_append(x_centers, sx_list, t_centers, st_list, xsh, sx, tk, st, min_sep=0.35 * shoulder_base)

    return np.array(x_centers), np.array(t_centers), np.array(sx_list), np.array(st_list)




def predictor_residual_profile_at_time(model, device, x0_test, nu_test, x_grid: np.ndarray, tk: float):
    pts = np.stack([x_grid, np.full_like(x_grid, tk)], axis=1)
    xt = torch.tensor(pts, dtype=torch.float32, device=device).view(1, -1, 2)
    xt.requires_grad_(True)
    p = torch.tensor([[x0_test, nu_test]], dtype=torch.float32, device=device)
    model.eval()
    u, _ = model(p, xt)
    grads = torch.autograd.grad(
        outputs=u,
        inputs=xt,
        grad_outputs=torch.ones_like(u),
        create_graph=False,
        retain_graph=False,
    )[0][0]
    u_np = u.detach().cpu().numpy().reshape(-1)
    ux_np = grads[:, 0].detach().cpu().numpy().reshape(-1)
    ut_np = grads[:, 1].detach().cpu().numpy().reshape(-1)
    r_np = ut_np + ux_np
    return u_np, ux_np, ut_np, r_np


def extract_refined_residual_centers_from_predictor(model, device, x0_test, nu_test, cfg: CorrectorConfig):
    t_vals = np.linspace(0.0, 1.0, cfg.K_t)
    dtg = t_vals[1] - t_vals[0] if cfg.K_t > 1 else 1.0
    x_fine = np.linspace(0.0, 1.0, cfg.fine_res_x, endpoint=False)
    dxf = 1.0 / max(cfg.fine_res_x, 2)

    x_centers, sx_list, t_centers, st_list = [], [], [], []
    st = max(cfg.sigma_t_res_scale * dtg, 2e-3)

    for tk in t_vals:
        U, Ux, Ut, R = predictor_residual_profile_at_time(model, device, x0_test, nu_test, x_fine, float(tk))
        amp = np.abs(U)
        amp_max = float(np.max(amp)) + 1e-12
        deriv_scale = 1e-6 + np.abs(Ux) + np.abs(Ut)
        support = (amp / amp_max) ** max(cfg.support_power, 1.0)
        score = np.abs(R) / deriv_scale
        score = score * (0.15 + 0.85 * support)

        cand = find_periodic_local_maxima(score)
        cand = sorted(cand.tolist(), key=lambda i: score[i], reverse=True)
        min_sep = max(5 * dxf, 0.28 * max(nu_test, dxf))
        selected = []
        for idx in cand:
            if amp[idx] < 0.10 * amp_max:
                continue
            xpk = float(x_fine[idx])
            if all(abs(periodic_distance_numpy(np.array([xpk]), x_fine[j])[0]) > min_sep for j in selected):
                selected.append(idx)
            if len(selected) >= cfg.residual_hotspots_per_time:
                break

        for idx in selected:
            xpk = float(x_fine[idx])
            sigma_hat = estimate_peak_sigma_from_profile(x_fine, amp, idx, nu_test)
            sigma_hat = clip_numpy(sigma_hat, max(1.2 * dxf, 0.18 * max(nu_test, dxf)), min(0.14, 1.8 * max(nu_test, 2*dxf)))

            shift = 0.55 * sigma_hat
            x_positions = [xpk, float((xpk - shift) % 1.0), float((xpk + shift) % 1.0)]
            spacing = local_periodic_spacing(np.array(x_positions), fallback=max(2.0 * dxf, 0.18 * max(nu_test, dxf)))
            center_base = clip_numpy(1.00 * spacing[0], max(1.5 * dxf, 0.16 * max(nu_test, dxf)), min(0.16, 0.95 * max(nu_test, 2*dxf), 1.8 * sigma_hat))

            # centered residual atoms with spacing-based overlap control
            for fac in cfg.residual_width_factors:
                sx = clip_numpy(fac * center_base, max(1.5 * dxf, 0.16 * max(nu_test, dxf)), min(0.18, 1.20 * max(nu_test, 2*dxf), 2.0 * center_base))
                unique_append(x_centers, sx_list, t_centers, st_list, xpk, sx, tk, st, min_sep=0.32 * center_base)

            # tiny symmetric offsets allow local recentering without characteristic priors
            for sign, idx_sp in zip((-1.0, 1.0), (1, 2)):
                xsh = float((xpk + sign * shift) % 1.0)
                side_base = clip_numpy(1.00 * spacing[idx_sp], max(1.5 * dxf, 0.17 * max(nu_test, dxf)), min(0.18, 1.10 * max(nu_test, 2*dxf), 1.9 * sigma_hat))
                sx = clip_numpy(1.05 * side_base, max(1.5 * dxf, 0.17 * max(nu_test, dxf)), min(0.18, 1.15 * max(nu_test, 2*dxf), 1.9 * side_base))
                unique_append(x_centers, sx_list, t_centers, st_list, xsh, sx, tk, st, min_sep=0.30 * side_base)

    return np.array(x_centers), np.array(t_centers), np.array(sx_list), np.array(st_list)


def build_background_geometry(cfg: CorrectorConfig):
    xg = np.linspace(0.0, 1.0, cfg.bg_x, endpoint=False)
    tg = np.linspace(0.0, 1.0, cfg.bg_t)
    dx = 1.0 / cfg.bg_x
    dt = 1.0 / max(cfg.bg_t - 1, 1)
    sx = cfg.bg_sigma_x_factor * dx * np.ones(cfg.bg_x * cfg.bg_t)
    st = cfg.sigma_t_bg_scale * dt * np.ones(cfg.bg_x * cfg.bg_t)
    Xg, Tg = np.meshgrid(xg, tg, indexing="ij")
    return Xg.ravel(), Tg.ravel(), sx, st


def build_corrector_geometry(model, device, x0_test, nu_test, cfg: CorrectorConfig):
    x1, t1, sx1, st1 = extract_dynamic_centers_from_predictor(model, device, x0_test, nu_test, cfg)
    x2, t2, sx2, st2 = extract_local_multiscale_centers_from_predictor(model, device, x0_test, nu_test, cfg)
    x3, t3, sx3, st3 = extract_refined_residual_centers_from_predictor(model, device, x0_test, nu_test, cfg)
    x4, t4, sx4, st4 = build_background_geometry(cfg)

    parts_x = [arr for arr in (x1, x2, x3, x4) if len(arr) > 0]
    parts_t = [arr for arr in (t1, t2, t3, t4) if len(arr) > 0]
    parts_sx = [arr for arr in (sx1, sx2, sx3, sx4) if len(arr) > 0]
    parts_st = [arr for arr in (st1, st2, st3, st4) if len(arr) > 0]

    mx = np.concatenate(parts_x) if parts_x else np.array([])
    mt = np.concatenate(parts_t) if parts_t else np.array([])
    sx = np.concatenate(parts_sx) if parts_sx else np.array([])
    st = np.concatenate(parts_st) if parts_st else np.array([])

    return {
        "mu_x": mx,
        "mu_t": mt,
        "sigma_x": sx,
        "sigma_t": st,
        "n_dynamic": len(x1),
        "n_refine": len(x2),
        "n_residual": len(x3),
        "n_background": len(x4),
        "n_hidden": len(mx),
        "cfg": cfg,
    }


def predictor_support_weight(model, device, x0_test, nu_test, X, T, cfg):
    U = predictor_field_numpy(model, device, x0_test, nu_test, X, T)
    umax = float(np.max(np.abs(U))) + 1e-12
    W = cfg.residual_weight_floor + (1.0 - cfg.residual_weight_floor) * (np.abs(U) / umax) ** cfg.support_power
    return U, W


class FrozenCorrector:
    def __init__(self, model, device, x0_test, nu_test, geometry, coeffs, alpha):
        self.model = model
        self.device = device
        self.x0_test = float(x0_test)
        self.nu_test = float(nu_test)
        self.geometry = geometry
        self.coeffs = coeffs.reshape(-1)
        self.alpha = float(alpha)

    def correction_on_grid(self, X, T):
        phi, _, _ = periodic_spacetime_basis_numpy(
            X.ravel(), T.ravel(),
            self.geometry["mu_x"], self.geometry["mu_t"],
            self.geometry["sigma_x"], self.geometry["sigma_t"],
        )
        w = phi @ self.coeffs
        return self.alpha * T * w.reshape(X.shape)

    def predict_on_grid(self, X, T):
        u_pred = predictor_field_numpy(self.model, self.device, self.x0_test, self.nu_test, X, T)
        return u_pred + self.correction_on_grid(X, T)



def fit_corrector(model, device, x0_test, nu_test, cfg: CorrectorConfig | None = None):
    if cfg is None:
        cfg = choose_corrector_config(nu_test)
    geom = build_corrector_geometry(model, device, x0_test, nu_test, cfg)
    if geom["n_hidden"] == 0:
        raise RuntimeError("Corrector geometry is empty.")

    # Coarse residual collocation.
    x = np.linspace(0.0, 1.0, cfg.colloc_x, endpoint=False)
    t_main = np.linspace(0.0, 1.0, cfg.colloc_t)
    X, T = np.meshgrid(x, t_main, indexing="ij")
    _, W = predictor_support_weight(model, device, x0_test, nu_test, X, T, cfg)
    _, Ux_pred, Ut_pred = predictor_derivatives_numpy(model, device, x0_test, nu_test, X, T)
    r_pred = Ut_pred + Ux_pred

    phi, phi_x, phi_t = periodic_spacetime_basis_numpy(
        X.ravel(), T.ravel(),
        geom["mu_x"], geom["mu_t"],
        geom["sigma_x"], geom["sigma_t"],
    )
    A_res = phi + T.ravel()[:, None] * (phi_t + phi_x)
    b_res = -r_pred.ravel()
    sw = np.sqrt(W.ravel())
    A_res = sw[:, None] * A_res
    b_res = sw * b_res

    # Fine residual collocation on predictor-derived guide times.
    t_guide = np.linspace(0.0, 1.0, cfg.K_t)
    x_fine = np.linspace(0.0, 1.0, cfg.fine_res_x, endpoint=False)
    Xf, Tf = np.meshgrid(x_fine, t_guide, indexing="ij")
    U_fine = np.zeros_like(Xf)
    R_fine = np.zeros_like(Xf)
    for k, tk in enumerate(t_guide):
        u_k, _ux_k, _ut_k, r_k = predictor_residual_profile_at_time(model, device, x0_test, nu_test, x_fine, float(tk))
        U_fine[:, k] = u_k
        R_fine[:, k] = r_k
    umax_f = float(np.max(np.abs(U_fine))) + 1e-12
    W_fine = cfg.residual_weight_floor + (1.0 - cfg.residual_weight_floor) * (np.abs(U_fine) / umax_f) ** max(cfg.support_power, 1.0)

    phi_f, phi_x_f, phi_t_f = periodic_spacetime_basis_numpy(
        Xf.ravel(), Tf.ravel(),
        geom["mu_x"], geom["mu_t"],
        geom["sigma_x"], geom["sigma_t"],
    )
    A_fine = phi_f + Tf.ravel()[:, None] * (phi_t_f + phi_x_f)
    b_fine = -R_fine.ravel()
    sw_fine = math.sqrt(cfg.fine_residual_boost) * np.sqrt(W_fine.ravel())
    A_fine = sw_fine[:, None] * A_fine
    b_fine = sw_fine * b_fine

    # Keep the correction small and concentrated near predictor support.
    xp = np.linspace(0.0, 1.0, cfg.probe_eval_x, endpoint=False)
    tp = np.linspace(0.0, 1.0, cfg.probe_eval_t)
    Xp, Tp = np.meshgrid(xp, tp, indexing="ij")
    Uprobe, Wprobe = predictor_support_weight(model, device, x0_test, nu_test, Xp, Tp, cfg)
    phi_p, phi_x_p, phi_t_p = periodic_spacetime_basis_numpy(
        Xp.ravel(), Tp.ravel(),
        geom["mu_x"], geom["mu_t"],
        geom["sigma_x"], geom["sigma_t"],
    )
    corr_field_map = Tp.ravel()[:, None] * phi_p
    support_sqrt = np.sqrt(Wprobe.ravel())[:, None]
    Bcorr = math.sqrt(cfg.corr_shrink) * (support_sqrt * corr_field_map)
    Bgrad = math.sqrt(0.06 * cfg.corr_shrink) * (support_sqrt * (Tp.ravel()[:, None] * phi_x_p))
    zcorr = np.zeros(Bcorr.shape[0])
    zgrad = np.zeros(Bgrad.shape[0])

    # Tiny anchor at final time near predictor support to prevent global drift.
    final_mask = np.isclose(Tp.ravel(), 1.0)
    Bfinal = math.sqrt(0.02 * cfg.corr_shrink) * (np.sqrt(Wprobe.ravel()[final_mask])[:, None] * corr_field_map[final_mask])
    zfinal = np.zeros(Bfinal.shape[0])

    # Precompute probe residual operator for alpha selection.
    _, Ux_probe, Ut_probe = predictor_derivatives_numpy(model, device, x0_test, nu_test, Xp, Tp)
    r_probe = Ut_probe + Ux_probe
    Aprobe = phi_p + Tp.ravel()[:, None] * (phi_t_p + phi_x_p)
    Wv = Wprobe.ravel()

    best = None
    for ridge in cfg.ridge_list:
        A_aug = np.vstack([
            A_res,
            A_fine,
            Bcorr,
            Bgrad,
            Bfinal,
            math.sqrt(cfg.coeff_shrink * ridge) * np.eye(geom["n_hidden"]),
        ])
        b_aug = np.concatenate([
            b_res,
            b_fine,
            zcorr,
            zgrad,
            zfinal,
            np.zeros(geom["n_hidden"]),
        ])
        coeffs, *_ = np.linalg.lstsq(A_aug, b_aug, rcond=1e-10)

        corr_field = corr_field_map @ coeffs
        corr_grad = (Tp.ravel()[:, None] * phi_x_p) @ coeffs
        corr_res = Aprobe @ coeffs
        for alpha in cfg.alpha_list:
            r_new = r_probe.ravel() + alpha * corr_res
            obj = (
                np.mean(Wv * (r_new ** 2))
                + 0.06 * np.mean(Wv * ((alpha * corr_field) ** 2))
                + 0.02 * np.mean(Wv * ((alpha * corr_grad) ** 2))
            )
            item = (obj, ridge, alpha, coeffs.copy())
            if (best is None) or (item[0] < best[0]):
                best = item

    _, best_ridge, best_alpha, best_coeffs = best
    geom["selected_ridge"] = float(best_ridge)
    geom["selected_alpha"] = float(best_alpha)
    return FrozenCorrector(model, device, x0_test, nu_test, geom, best_coeffs, best_alpha)

# ============================================================
# Plotting / evaluation with corrector
# ============================================================
def evaluate_case_on_grid(model, device, x0_test, nu_test, Nx=200, Nt=200, add_corrector=False):
    x = np.linspace(0.0, 1.0, Nx, endpoint=False)
    t = np.linspace(0.0, 1.0, Nt)
    X, T = np.meshgrid(x, t, indexing="ij")
    u_exact = exact_solution(X, T, x0_test, nu_test)
    u_pred = predictor_field_numpy(model, device, x0_test, nu_test, X, T)
    abs_err_pred = np.abs(u_pred - u_exact)
    rel_l2_pred = np.linalg.norm((u_pred - u_exact).ravel()) / (np.linalg.norm(u_exact.ravel()) + 1e-14)

    out = {
        "u_exact": u_exact,
        "u_pred": u_pred,
        "abs_err_pred": abs_err_pred,
        "rel_l2_pred": rel_l2_pred,
    }
    if add_corrector:
        corr_cfg = choose_corrector_config(nu_test)
        corrector = fit_corrector(model, device, x0_test, nu_test, corr_cfg)
        u_corr = corrector.predict_on_grid(X, T)
        abs_err_corr = np.abs(u_corr - u_exact)
        rel_l2_corr = np.linalg.norm((u_corr - u_exact).ravel()) / (np.linalg.norm(u_exact.ravel()) + 1e-14)
        out.update({
            "u_corr": u_corr,
            "abs_err_corr": abs_err_corr,
            "rel_l2_corr": rel_l2_corr,
            "corrector": corrector,
        })
    return out


def plot_predictor_corrector_fields(model, device, test_cases, save_dir, Nx=220, Nt=220):
    num_tests = len(test_cases)
    fig, axs = plt.subplots(4, num_tests, figsize=(4.4 * num_tests, 11.8), constrained_layout=True)
    if num_tests == 1:
        axs = axs.reshape(4, 1)

    results = []
    exacts, preds, corrs, errs = [], [], [], []
    for name, x0, nu in test_cases:
        out = evaluate_case_on_grid(model, device, x0, nu, Nx=Nx, Nt=Nt, add_corrector=True)
        exacts.append((name, x0, nu, out["u_exact"]))
        preds.append((name, x0, nu, out["u_pred"]))
        corrs.append((name, x0, nu, out["u_corr"]))
        errs.append((name, x0, nu, out["abs_err_corr"]))
        results.append({
            "name": name,
            "x0": x0,
            "nu": nu,
            "rel_l2_pred": out["rel_l2_pred"],
            "rel_l2_corr": out["rel_l2_corr"],
            "n_hidden": out["corrector"].geometry["n_hidden"],
            "n_dynamic": out["corrector"].geometry["n_dynamic"],
            "n_refine": out["corrector"].geometry["n_refine"],
            "n_residual": out["corrector"].geometry["n_residual"],
            "n_background": out["corrector"].geometry["n_background"],
        })

    sol_vals = np.concatenate([u.ravel() for *_, u in exacts + preds + corrs])
    err_vals = np.concatenate([u.ravel() for *_, u in errs])
    sol_vmin, sol_vmax = float(sol_vals.min()), float(sol_vals.max())
    err_vmax = float(err_vals.max())

    sol_images = []
    err_images = []

    for j in range(num_tests):
        name, x0, nu, ue = exacts[j]
        _, _, _, up = preds[j]
        _, _, _, uc = corrs[j]
        _, _, _, er = errs[j]

        ax = axs[0, j]
        im0 = ax.imshow(
            ue.T, origin="lower", extent=[0, 1, 0, 1],
            aspect="auto", vmin=sol_vmin, vmax=sol_vmax, cmap="viridis"
        )
        sol_images.append(im0)
        ax.set_title(rf"$(x_0,\nu)=({x0:.2f},{nu:.2f})$")
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel("Exact", labelpad=8)

        ax = axs[1, j]
        im1 = ax.imshow(
            up.T, origin="lower", extent=[0, 1, 0, 1],
            aspect="auto", vmin=sol_vmin, vmax=sol_vmax, cmap="viridis"
        )
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel(r"$u_{pred}$", labelpad=8)

        ax = axs[2, j]
        im2 = ax.imshow(
            uc.T, origin="lower", extent=[0, 1, 0, 1],
            aspect="auto", vmin=sol_vmin, vmax=sol_vmax, cmap="viridis"
        )
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel(r"$u_{corr}$", labelpad=8)

        ax = axs[3, j]
        im3 = ax.imshow(
            er.T, origin="lower", extent=[0, 1, 0, 1],
            aspect="auto", vmin=0.0, vmax=err_vmax, cmap="magma"
        )
        err_images.append(im3)
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel(r"$|u_{corr}-u_{exact}|$", labelpad=8)

    cbar1 = fig.colorbar(
        sol_images[0],
        ax=axs[0:3, :],
        fraction=0.025,
        pad=0.015,
        shrink=0.95
    )
    cbar1.set_label(r"$u(x,t)$")

    cbar2 = fig.colorbar(
        err_images[0],
        ax=axs[3, :],
        fraction=0.025,
        pad=0.015,
        shrink=0.92
    )
    cbar2.set_label("Absolute Error")

    fig.suptitle("Linear advection: exact vs predictor vs corrector", y=1.02, fontsize=15)
    out_path = os.path.join(save_dir, "predictor_corrector_multicase_fields.png")
    fig.savefig(out_path, dpi=250, bbox_inches="tight")
    plt.close(fig)
    return results


def plot_training_history(history, save_path):
    plt.figure(figsize=(7, 4.5))
    for key in ["loss", "loss_pde", "loss_ic", "loss_per", "loss_center", "loss_width"]:
        plt.plot(history["epoch"], history[key], lw=1.8, label=key.replace("loss_", "").upper())
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training history")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=250)
    plt.close()


def save_summary_csv(summary, save_dir):
    path = os.path.join(save_dir, "test_case_summary_with_corrector.csv")
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["name", "x0", "nu", "rel_l2_pred", "rel_l2_corr", "n_hidden", "n_dynamic", "n_refine", "n_residual", "n_background"],
        )
        writer.writeheader()
        for row in summary:
            writer.writerow(row)


def visualize_dynamic_centers(model, device, test_cases, save_dir, K=60):
    model.eval()
    t_vals = np.linspace(0.0, 1.0, K)
    for name, x0, nu in test_cases:
        p = torch.tensor([[x0, nu]], dtype=torch.float32, device=device)
        t = torch.tensor(t_vals, dtype=torch.float32, device=device).view(1, K, 1)
        with torch.no_grad():
            alpha, xi, h, gate = model.basis_params(p, t)
        xi_np = xi[0].cpu().numpy()
        h_np = h[0].cpu().numpy()
        w_np = np.abs((alpha[0] * gate[0]).cpu().numpy())

        X_pts = xi_np.reshape(-1)
        T_pts = np.repeat(t_vals, model.M)
        S_pts = np.clip(120.0 * h_np.reshape(-1), 8.0, 80.0)
        C_pts = w_np.reshape(-1)

        fig, ax = plt.subplots(figsize=(5.5, 5.0))
        sc = ax.scatter(X_pts, T_pts, c=C_pts, s=S_pts, cmap="viridis", alpha=0.85)
        plt.colorbar(sc, ax=ax, label=r"$|\alpha_j g_j|$")
        ax.scatter([x0], [0.0], color="red", marker="x", s=90, label="IC center")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xlabel("x")
        ax.set_ylabel("t")
        ax.set_title(f"Dynamic basis centers: {name}")
        ax.legend(loc="upper right")
        fig.tight_layout()
        fig.savefig(os.path.join(save_dir, f"dynamic_centers_{name}.png"), dpi=250)
        plt.close(fig)


def plot_corrector_geometry(model, device, test_cases, save_dir):
    for name, x0, nu in test_cases:
        cfg = choose_corrector_config(nu)
        geom = build_corrector_geometry(model, device, x0, nu, cfg)
        fig, ax = plt.subplots(figsize=(5.8, 5.0))
        n1 = geom["n_dynamic"]
        n2 = geom["n_refine"]
        n3 = geom["n_residual"]
        ax.scatter(geom["mu_x"][:n1], geom["mu_t"][:n1], s=np.clip(150 * geom["sigma_x"][:n1], 10, 90), label="predictor-guided", alpha=0.8)
        ax.scatter(geom["mu_x"][n1:n1+n2], geom["mu_t"][n1:n1+n2], s=np.clip(200 * geom["sigma_x"][n1:n1+n2], 12, 75), marker="x", label="local multiscale", alpha=0.8)
        ax.scatter(geom["mu_x"][n1+n2:n1+n2+n3], geom["mu_t"][n1+n2:n1+n2+n3], s=np.clip(200 * geom["sigma_x"][n1+n2:n1+n2+n3], 10, 70), marker="^", label="fine residual", alpha=0.8)
        ax.scatter(geom["mu_x"][n1+n2+n3:], geom["mu_t"][n1+n2+n3:], s=np.clip(120 * geom["sigma_x"][n1+n2+n3:], 8, 60), marker="s", label="background", alpha=0.55)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xlabel("x")
        ax.set_ylabel("t")
        ax.set_title(f"Corrector geometry: {name} (n={geom['n_hidden']})")
        ax.legend(loc="upper right")
        fig.tight_layout()
        fig.savefig(os.path.join(save_dir, f"corrector_geometry_{name}.png"), dpi=250)
        plt.close(fig)


# ============================================================
# Training
# ============================================================
@dataclass
class TrainConfig:
    B_pde: int = 4
    N_int: int = 256
    N_ic: int = 128
    N_per: int = 128
    N_near: int = 96
    epochs: int = 5000
    lr: float = 1e-3
    print_every: int = 100
    nu_min: float = 0.03
    nu_max: float = 0.12
    nu_easy: float = 0.06
    lambda_ic: float = 10.0
    lambda_per: float = 1.0
    lambda_center: float = 5e-4
    lambda_width: float = 2e-4


def train_meta_advection(model: nn.Module, device: torch.device, cfg: TrainConfig):
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
    best_loss = float("inf")
    best_state = None
    history = {k: [] for k in ["epoch", "loss", "loss_pde", "loss_ic", "loss_per", "loss_center", "loss_width"]}

    for ep in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad()

        if ep <= cfg.epochs // 2:
            nu_min_curr, nu_max_curr = cfg.nu_easy, cfg.nu_max
        else:
            nu_min_curr, nu_max_curr = cfg.nu_min, cfg.nu_max

        p_batch = sample_p_batch(cfg.B_pde, device, nu_range=(nu_min_curr, nu_max_curr))
        xt_int = sample_interior_points_batch(cfg.B_pde, cfg.N_int, device)
        xt_ic = sample_ic_points_batch(cfg.B_pde, cfg.N_ic, device)
        xtL, xtR = sample_periodic_points_batch(cfg.B_pde, cfg.N_per, device)
        xt_near = sample_near_ic_points_batch(cfg.B_pde, cfg.N_near, device, t_max=0.15)

        res_int = advection_residual_batch(model, p_batch, xt_int)
        res_near = advection_residual_batch(model, p_batch, xt_near)
        loss_pde = torch.mean(res_int ** 2) + 0.5 * torch.mean(res_near ** 2)

        u_ic, _ = model(p_batch, xt_ic)
        x_ic = xt_ic[:, :, 0]
        x0 = p_batch[:, 0:1].expand_as(x_ic)
        nu = p_batch[:, 1:2].expand_as(x_ic)
        u_ic_true = periodic_gaussian_torch(x_ic, x0, nu)
        loss_ic = torch.mean((u_ic - u_ic_true) ** 2)

        uL, _ = model(p_batch, xtL)
        uR, _ = model(p_batch, xtR)
        loss_per = torch.mean((uL - uR) ** 2)

        _, (gate_aux, alpha_aux, xi_aux, h_aux) = model(p_batch, xt_near)
        dxi_dt = xi_aux[:, 1:, :] - xi_aux[:, :-1, :] if xi_aux.shape[1] > 1 else xi_aux * 0.0
        loss_center = torch.mean(torch.abs(dxi_dt)) + 0.2 * torch.mean(torch.abs(gate_aux * alpha_aux))
        target_h = p_batch[:, 1:2].unsqueeze(1)
        loss_width = torch.mean((h_aux - torch.clamp(1.25 * target_h, model.h_min, model.h_max)) ** 2)

        loss = (
            loss_pde
            + cfg.lambda_ic * loss_ic
            + cfg.lambda_per * loss_per
            + cfg.lambda_center * loss_center
            + cfg.lambda_width * loss_width
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        history["epoch"].append(ep)
        history["loss"].append(loss.item())
        history["loss_pde"].append(loss_pde.item())
        history["loss_ic"].append(loss_ic.item())
        history["loss_per"].append(loss_per.item())
        history["loss_center"].append(loss_center.item())
        history["loss_width"].append(loss_width.item())

        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if ep % cfg.print_every == 0 or ep == 1:
            print(
                f"Epoch {ep:4d} | Loss {loss.item():.3e} | "
                f"PDE {loss_pde.item():.3e} | IC {loss_ic.item():.3e} | PER {loss_per.item():.3e}"
            )

        if device.type == "cuda":
            torch.cuda.empty_cache()

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history



# ============================================================
# Main: train once and save model
# ============================================================
def main():
    parser = argparse.ArgumentParser(description="Dynamic-basis meta-SPINN training only")
    parser.add_argument("--gpu", action="store_true", help="Use GPU if available")
    parser.add_argument("--epochs", type=int, default=5000)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--M", type=int, default=32)
    parser.add_argument("--outdir", type=str, default="TC_02_FIGURES")
    parser.add_argument("--model_path", type=str, default="meta_spinn_dynamic_basis.pt")
    args, _unknown = parser.parse_known_args()

    os.makedirs(args.outdir, exist_ok=True)
    device = get_device(args.gpu)
    print("Using device:", device)

    cfg = TrainConfig(epochs=args.epochs, lr=args.lr)
    model = DynamicMetaSPINN(M=args.M).to(device)
    model, history = train_meta_advection(model, device, cfg)

    model_path = os.path.join(args.outdir, args.model_path)
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to: {model_path}")

    history_path = os.path.join(args.outdir, "training_history.npz")
    np.savez_compressed(
        history_path,
        epoch=np.asarray(history["epoch"]),
        loss=np.asarray(history["loss"]),
        loss_pde=np.asarray(history["loss_pde"]),
        loss_ic=np.asarray(history["loss_ic"]),
        loss_per=np.asarray(history["loss_per"]),
        loss_center=np.asarray(history["loss_center"]),
        loss_width=np.asarray(history["loss_width"]),
    )
    print(f"Training history saved to: {history_path}")

    plot_training_history(history, os.path.join(args.outdir, "training_loss.png"))


if __name__ == "__main__":
    main()

Using device: cpu
Epoch    1 | Loss 6.045e+00 | PDE 4.232e+00 | IC 1.813e-01 | PER 2.125e-18
Epoch  100 | Loss 4.999e-01 | PDE 2.150e-01 | IC 2.849e-02 | PER 6.103e-17
Epoch  200 | Loss 2.691e-01 | PDE 1.733e-01 | IC 9.573e-03 | PER 3.135e-16
Epoch  300 | Loss 1.285e-01 | PDE 1.014e-01 | IC 2.702e-03 | PER 6.324e-16
Epoch  400 | Loss 1.124e-01 | PDE 8.960e-02 | IC 2.282e-03 | PER 5.696e-16
Epoch  500 | Loss 8.354e-02 | PDE 6.591e-02 | IC 1.761e-03 | PER 1.017e-15
Epoch  600 | Loss 5.594e-02 | PDE 4.204e-02 | IC 1.387e-03 | PER 1.026e-15
Epoch  700 | Loss 4.860e-02 | PDE 2.860e-02 | IC 1.998e-03 | PER 9.860e-16
Epoch  800 | Loss 8.857e-02 | PDE 7.985e-02 | IC 8.691e-04 | PER 3.105e-15
Epoch  900 | Loss 4.522e-02 | PDE 4.225e-02 | IC 2.952e-04 | PER 3.384e-15
Epoch 1000 | Loss 1.607e-02 | PDE 1.159e-02 | IC 4.452e-04 | PER 1.914e-15
Epoch 1100 | Loss 5.483e-02 | PDE 3.488e-02 | IC 1.992e-03 | PER 3.153e-16
Epoch 1200 | Loss 2.254e-02 | PDE 1.892e-02 | IC 3.596e-04 | PER 3.030e-15
Epoch 1